# 01 — Exploratory data analysis: Southern Italy — A = Rome, B = Sicily 


## Підсумок EDA: що ми знаємо і чого ще не знаємо

**Дані:** Inside Airbnb, Rome (знімок 2026-09-17) і Sicily (знімок 2026-09-25). Один рядок = одне оголошення, `id` унікальний.  
Rome: 37 451 оголошення (15 муніципіїв). Sicily: 56 783 оголошення (369 комун). У кожному місті 91 колонка, типи збігаються, parse failures = 0.

### ✅ Що ми тепер знаємо

**Якість і структура даних**
- `price` — це рядок `"$1,000.00"`, але валюта насправді EUR. Парсимо явно.
- Частина колонок порожня на 100% в обох містах (`host_since`, `bathrooms`, `host_response_*`, `instant_bookable`, `neighbourhood`, …). Замінники: `bathrooms_text` і `hosts_time_as_host_*`.
- `review_scores_* = 0` у Sicily виходить за шкалу 1–5, тому перекодовуємо в null.

**Пропуски мають патерн, вони не випадкові**
- **Ціни немає в 9.4% (Rome) і 10.7% (Sicily) оголошень. Це MNAR.** Здебільшого це неактивні оголошення: у 70% / 49% з них 0 вільних днів, а з останнього відгуку минуло близько року. Їх виключаємо з навчання і не імпутуємо.
- **Пропуски у відгуках = оголошення без жодного відгуку** (збіг 100%, 11.9% / 24.8% оголошень). Це категорія: замість імпутації додаємо ознаку `no_reviews`.
- `minimum_nights` порожній лише в оголошеннях із закритим календарем.

**Ціна**
- Розподіл сильно скошений: skew 30 / 113, а для log(price) лише 0.8 / 1.1. Звітуємо медіану, модель вчимо на `log(price)`.
- Медіана: **Rome 188 €, Sicily 115 €**. Rome дорожчий у кожному `room_type` (у 1.5–1.7 раза). **Simpson's paradox немає.**
- Найсильніший чинник ціни — розмір житла (`accommodates`, `bedrooms`, `bathrooms`, ρ ≈ 0.4–0.5). Ці ознаки сильно корелюють між собою.
- Локація важлива, але діє по-різному: у Rome вирішує близькість до центру, у Sicily — узбережжя й острови проти внутрішніх районів. Сира широта в Sicily має ρ = 0.
- Long-stay (`minimum_nights ≥ 28`, 5% / 12% оголошень) — окремий ринок, дешевший за ніч. `long_stay` має бути ознакою моделі.
- Попит і активність (відгуки, availability, occupancy) майже не корелюють з ціною (|ρ| < 0.1).

**Викиди**
- Явних помилок немає. 81 `suspicious` оголошення (≥ 5 000 € або < 10 €) — це ціни-«заглушки», їх виключаємо. 1 049 / 2 117 оголошень `tail` (вілли, великі апартаменти) лишаємо.
- Межі IQR рахуємо на log(price) і **окремо для кожного міста**: верхня межа 789 € у Rome, 505 € у Sicily.

**Leakage**
- `estimated_revenue_l365d / estimated_occupancy_l365d = price` у 100% оголошень. Кореляція цього не показує, видно лише з того, як колонку обчислено. Також виключаємо `price_quote_*`, `price_per_guest`, ознаки попиту, ідентифікатори й константи знімка.
- 31–33% оголошень мають «близнюка» того самого хоста, тому split групуємо за `host_id`.

### ❓ Чого ми ще не знаємо

- **Чи перенесеться все це на Naples.** Типи `inferSchema`, частка неактивних оголошень, назви районів і рівень цін там можуть бути іншими. Усі висновки — лише про Rome і Sicily.
- **Сезонність.** Ми бачили тільки один знімок кінця вересня. Ціни й доступність у високий сезон невідомі (особливо для курортів Sicily, де 21% календарів відкриті майже весь рік).
- **Реальні ціни бронювань.** `price` — це ціна в оголошенні на дату скрейпу, а не фактична ціна, за якою бронюють.
- **Чи працюють ознаки локації, які ми плануємо** (відстань до центру, до узбережжя чи туристичних центрів). Поки це гіпотеза: ми не рахували ці ознаки й не перевіряли їх.
- **Наскільки точною буде модель.** EDA показує лише кореляції. Очікуємо, що в Sicily похибка буде більшою (ринок неоднорідний, зв'язки слабші)
- **Межа `suspicious`** (5 000 € / 10 €) підібрана вручну за двома містами. Чи підійде вона іншим ринкам, невідомо.


## 1. Параметри 

Регіон **Southern Italy (EUR)**: A = **Rome**, B = **Sicily**, 



In [0]:
# --- PARAMETERS -------------------------------------------------------------

CITY_SOURCES = {
    "rome":   ("italy/lazio/rome",     "2026-09-17"),
    "sicily": ("italy/sicilia/sicily", "2026-09-25"),
}

CURRENCY = "EUR"

SEED          = 42             # для всіх випадкових кроків
SAMPLE_N      = 30_000         # максимум рядків на місто, що йдуть у pandas (кореляції)
IQR_K         = 1.5            # Tukey fence
MZ_THRESHOLD  = 3.5            # modified z-score (Iglewicz & Hoaglin)
MIN_PRICE     = 10             # EUR/ніч: нижче — «підозріло дешево»
MAX_PRICE     = 5_000          # EUR/ніч: вище — «підозріло дорого» (у Сицилії є справжні вілли за 1–3 тис.)
MAX_GUESTS    = 16             # Airbnb не дозволяє accommodates > 16
LONG_STAY     = 28             # minimum_nights >= 28 — ринок довгострокової оренди
# ---------------------------------------------------------------------------


In [0]:
import os, re, datetime, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

for city, (_, snap) in CITY_SOURCES.items():
    assert re.fullmatch(r"\d{4}-\d{2}-\d{2}", snap), f"Вкажіть дату знімка для {city} (зараз '{snap}')"

CITY_ORDER = list(CITY_SOURCES)
COLORS = dict(zip(CITY_ORDER, ["#2a6fdb", "#e07b00"]))  
GREY = "#9a9a9a"
KEY = {}   # ключові цифри для підсумку


## 2. Завантаження: один знімок на місто, один раз, у власну схему



In [0]:
user_name = (spark.sql("SELECT current_user()").first()[0].split("@")[0]
             .replace(".", "_").replace("-", "_"))
schema = f"lab_{user_name}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{schema}")
spark.sql(f"USE workspace.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.{schema}.raw")
T = lambda name: f"workspace.{schema}.{name}"

CITIES = []
for city, (path, snap) in CITY_SOURCES.items():
    url = f"https://data.insideairbnb.com/{path}/{snap}/data/listings.csv.gz"
    raw_file = f"/Volumes/workspace/{schema}/raw/{city}_{snap}_listings.csv.gz"
    table = f"{city}_listings_raw"
    if not spark.catalog.tableExists(table):
        if not os.path.exists(raw_file):
            try:
                urllib.request.urlretrieve(url, raw_file)
            except Exception as e:
                raise RuntimeError(f"{url}: {e}. Перевірте дату знімка на insideairbnb.com/get-the-data "
                                   f"або завантажте файл вручну у {raw_file}")
        (spark.read
            .option("header", True)
            .option("multiLine", True)
            .option("escape", '"')
            .option("inferSchema", True)
            .csv(raw_file)
            .withColumn("snapshot_date", F.lit(snap).cast("date"))
            .write.saveAsTable(table))
    version = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]
    CITIES.append({"city": city, "snapshot": snap, "url": url, "table": T(table), "delta_version": version,
                   "rows": spark.table(table).count()})
display(pd.DataFrame(CITIES))


Для EDA усі колонки приводимо до `string`: `inferSchema` вгадує типи по-різному в різних містах (саме це зламається на місті C),
а профіль має показати, що насправді лежить у файлі (`price` = `"$1,234.00"`). Типізуємо далі самі, явними правилами.


In [0]:
def read_raw(c):
    t = spark.read.option("versionAsOf", c["delta_version"]).table(c["table"])   # pin version (L6 Step 8)
    t = t.drop(*[x for x in ("city", "snapshot_date") if x in t.columns])
    t = t.select([F.col(f"`{x}`").cast("string").alias(x) for x in t.columns])
    return t.withColumn("city", F.lit(c["city"])).withColumn("snapshot_date", F.to_date(F.lit(c["snapshot"])))

raw_by_city = {c["city"]: read_raw(c) for c in CITIES}

a, b = CITY_ORDER
types = {c["city"]: dict(spark.table(c["table"]).dtypes) for c in CITIES}
print(f"Колонок: {a}={len(types[a])}, {b}={len(types[b])}")
print(f"Є тільки в {a}:", sorted(set(types[a]) - set(types[b])))
print(f"Є тільки в {b}:", sorted(set(types[b]) - set(types[a])))
diff = [(col, types[a][col], types[b][col]) for col in types[a] if col in types[b] and types[a][col] != types[b][col]]
if diff:
    display(pd.DataFrame(diff, columns=["column", f"type_{a}", f"type_{b}"]))
else:
    print("inferSchema визначив однакові типи для всіх спільних колонок")

raw = raw_by_city[a].unionByName(raw_by_city[b], allowMissingColumns=True)


## 3. Опис датасету: дата знімка, grain, кількість записів (Step 1–2)

Перевіряємо, що **один рядок = одне оголошення**: `id` має бути унікальним і числовим. Нечислові `id` означають зламаний парсинг CSV (зсунуті колонки через переноси рядків).


In [0]:
desc = (raw.groupBy("city", "snapshot_date").agg(
            F.count("*").alias("rows"),
            F.countDistinct("id").alias("distinct_id"),
            F.sum(F.col("id").isNull().cast("int")).alias("null_id"),
            F.sum((~F.col("id").rlike(r"^\d+$")).cast("int")).alias("non_numeric_id"),
            F.countDistinct("host_id").alias("distinct_hosts"),
            F.min("last_scraped").alias("first_scraped"),
            F.max("last_scraped").alias("last_scraped"),
        ).orderBy("city"))
display(desc)

d = desc.toPandas().set_index("city")
for city in CITY_ORDER:
    KEY[f"rows_{city}"] = int(d.loc[city, "rows"])
    KEY[f"snapshot_{city}"] = str(d.loc[city, "snapshot_date"])
    KEY[f"scraped_{city}"] = f"{d.loc[city,'first_scraped']} … {d.loc[city,'last_scraped']}"
    if d.loc[city, "rows"] != d.loc[city, "distinct_id"]:
        print(f"⚠ {city}: id не унікальний — перевірте дублікати/парсинг")
print("Grain: один рядок = одне оголошення на дату знімка" if (d["rows"] == d["distinct_id"]).all() else "Grain під питанням")


### 3.1 Профіль усіх колонок «як є» 


In [0]:
def profile(df):
    """One row per column: type, % null, distinct count, min, max (L6)."""
    return (df.agg(*[
                F.struct(
                    F.lit(c).alias("column"), F.lit(t).alias("type"),
                    F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 2).alias("null_pct"),
                    F.approx_count_distinct(c).alias("distinct"),
                    F.min(c).cast("string").alias("min"),
                    F.max(c).cast("string").alias("max"),
                ).alias(c) for c, t in df.dtypes])
            .select(F.explode(F.array(*df.columns)).alias("p")).select("p.*"))

for c in CITIES:
    print(c["city"])
    display(profile(spark.read.option("versionAsOf", c["delta_version"]).table(c["table"])))

%md
### Висновки з профілю (Rome — знімок 2026-09-17, Sicily — знімок 2026-09-25)

**Датасет**
- Один рядок = одне оголошення, у кожній таблиці один знімок (`scrape_id` має одне значення). Rome ≈ 36 тис., Sicily ≈ 56 тис. оголошень (точні числа див. у розділі 3).
- Rome поділено на 15 муніципіїв, Sicily — на 369 комун. Тому `neighbourhood_cleansed` не можна порівнювати між містами, і як ознака він не перенесеться на Naples. Для локації беремо `latitude`/`longitude`.

**Що мусить робити пайплайн**
- `price` має тип `string` (`"$1,000.00"`), тому парсимо його явно. Валюта — EUR, попри символ `$`.
- 13 колонок порожні на 100% в обох містах (`host_since`, `bathrooms`, `host_response_rate`, `instant_bookable`, `neighbourhood` …). Їх видаляємо.
- `bathrooms` беремо з `bathrooms_text`, стаж хоста — з `hosts_time_as_host_years/months` (бо `host_since` порожній).
- `review_scores_*` = 0 у Sicily виходять за шкалу 1–5, тому перекодовуємо в null. Перевіряємо діапазон значень, а не орієнтуємося на те, що бачили в одному місті.

**Пропуски, які впливають на модель**
- Ціни немає в 9.4% (Rome) і 10.7% (Sicily) рядків. Ці рядки виключаємо з train/test, бо в них немає таргету.
- Відгуків немає в 11.9% (Rome) і 24.8% (Sicily) рядків: у них одночасно порожні `review_scores_*`, `first/last_review`, `reviews_per_month`. Це категорія «без відгуків», тому не імпутуємо, а додаємо індикатор.
- `minimum_nights` порожній у ~3% рядків, `beds` — у ~1.8%. Імпутуємо й додаємо флаг.

**Викиди**
- Ціна за ніч доходить до 37 тис. € у Rome і 90 тис. € у Sicily. Ймовірно, це ціни-«заглушки». 
- `minimum_nights` доходить до 720–1 124 ночей: це довгострокова оренда, тому додаємо ознаку `long_stay`.

**Leakage — заборонені колонки**
- `price_quote_price_per_night`, `price_quote_total_price`, `price_quote_raw` — це та сама ціна, тільки з котирування.
- `estimated_revenue_l365d` порожня рівно там, де порожня `price`, тобто обчислена з ціни.
- `estimated_occupancy_l365d`, `number_of_reviews*`, `review_scores_*` — наслідок ціни, і для нового оголошення їх ще немає.
- `id`, `host_id`, URL-колонки, `scrape_id`, `last_scraped` — ідентифікатори й константи знімка, а не ознаки.

## 4. Типізація і перевірка, що нічого не загубилось

Кожна колонка має правило парсингу. `try_cast` (а не `cast`) — бо serverless працює в ANSI-режимі і звичайний `cast("abc")` падає.
Після парсингу рахуємо **parse failures**: значення було, а після типізації стало `null`. Це те, що пайплайн мусить обробляти явно.


In [0]:
def q(c): return f"`{c}`"
def has(c): return c in raw.columns

NUMERIC = ["latitude", "longitude", "accommodates", "bathrooms", "bedrooms", "beds",
           "minimum_nights", "maximum_nights", "availability_30", "availability_60", "availability_90",
           "availability_365", "availability_eoy", "number_of_reviews", "number_of_reviews_ltm",
           "number_of_reviews_l30d", "number_of_reviews_ly", "reviews_per_month",
           "host_listings_count", "host_total_listings_count", "calculated_host_listings_count",
           "calculated_host_listings_count_entire_homes", "calculated_host_listings_count_private_rooms",
           "calculated_host_listings_count_shared_rooms",
           "estimated_occupancy_l365d", "estimated_revenue_l365d",
           "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
           "review_scores_checkin", "review_scores_communication", "review_scores_location",
           "review_scores_value", "hosts_time_as_host_years", "hosts_time_as_host_months",
            "hosts_time_as_user_years", "hosts_time_as_user_months"]
MONEY   = ["price"]
PERCENT = ["host_response_rate", "host_acceptance_rate"]
BOOL    = ["host_is_superhost", "host_has_profile_pic", "host_identity_verified",
           "has_availability", "instant_bookable"]
DATES   = ["last_scraped", "host_since", "first_review", "last_review", "calendar_last_scraped"]
STRINGS = ["room_type", "property_type", "neighbourhood_cleansed", "neighbourhood_group_cleansed",
           "host_response_time", "bathrooms_text", "license", "name", "description", "amenities",
           "calendar_updated"]

SPEC = {}   # output column -> (raw column, typed expression)
SPEC["id"] = ("id", F.expr("try_cast(id AS BIGINT)"))
SPEC["host_id"] = ("host_id", F.expr("try_cast(host_id AS BIGINT)"))
for c in NUMERIC:
    if has(c): SPEC[c] = (c, F.expr(f"try_cast({q(c)} AS DOUBLE)"))
for c in MONEY:
    if has(c): SPEC[c] = (c, F.expr(f"try_cast(regexp_replace({q(c)}, '[^0-9.]', '') AS DOUBLE)"))
for c in PERCENT:
    if has(c): SPEC[c] = (c, F.expr(f"try_cast(regexp_replace({q(c)}, '%', '') AS DOUBLE) / 100"))
for c in BOOL:
    if has(c): SPEC[c] = (c, F.expr(f"CASE WHEN lower({q(c)}) IN ('t', 'true') THEN true WHEN lower({q(c)}) IN ('f', 'false') THEN false END"))
for c in DATES:
    if has(c): SPEC[c] = (c, F.expr(f"try_cast(left({q(c)}, 10) AS DATE)"))
for c in STRINGS:
    if has(c): SPEC[c] = (c, F.expr(f"nullif(trim({q(c)}), '')"))

# parse failures: було непорожнє значення → стало null
fail = raw.agg(*[
    F.sum(F.when(F.col(rc).isNotNull() & (F.trim(F.col(rc)) != "") & e.isNull(), 1).otherwise(0)).alias(out)
    for out, (rc, e) in SPEC.items()]).toPandas().T.rename(columns={0: "parse_failures"})
fail = fail[fail.parse_failures > 0].sort_values("parse_failures", ascending=False)
if len(fail):
    display(fail.reset_index().rename(columns={"index": "column"}))
else:
    print("Parse failures: 0 — усі непорожні значення успішно типізовано")


In [0]:
typed = raw.select("city", "snapshot_date", *[e.alias(out) for out, (rc, e) in SPEC.items()])

# похідні колонки, які знадобляться і в EDA, і в пайплайні
if "bathrooms_text" in typed.columns:
    typed = (typed
        .withColumn("bathrooms_from_text", F.expr("""
            CASE WHEN lower(bathrooms_text) LIKE '%half-bath%' THEN 0.5
                 ELSE try_cast(regexp_extract(bathrooms_text, '([0-9]+[.]?[0-9]*)', 1) AS DOUBLE) END"""))
        .withColumn("bathrooms_shared", F.lower("bathrooms_text").contains("shared")))
typed = (typed
    .withColumn("log_price", F.when(F.col("price") > 0, F.log("price")))
    .withColumn("price_per_guest", F.when(F.col("accommodates") > 0, F.col("price") / F.col("accommodates")))
    .withColumn("amenities_count", F.size(F.from_json("amenities", "array<string>")))
    .withColumn("days_since_last_review", F.datediff("snapshot_date", "last_review"))
    .withColumn("host_tenure_months", F.col("hosts_time_as_host_years") * 12 + F.col("hosts_time_as_host_months"))
    .withColumn("long_stay", F.col("minimum_nights") >= LONG_STAY))

typed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(T("eda_silver_listings_typed"))
df = spark.table(T("eda_silver_listings_typed"))
print(df.count(), "rows,", len(df.columns), "columns")
df.printSchema()


In [0]:
if "bathrooms" in df.columns and "bathrooms_from_text" in df.columns:
    display(df.groupBy("city").agg(
        F.avg(F.col("bathrooms").isNull().cast("int")).alias("bathrooms_null"),
        F.avg(F.col("bathrooms_from_text").isNull().cast("int")).alias("bathrooms_from_text_null")))
        


## 5. Пропуски: скільки, де, і за яким патерном

**< 1%** — можна видалити рядки; **1–30%** — імпутувати + індикатор `is_missing`; **> 30%** — подумати про видалення колонки, спершу зрозумівши *чому* порожньо; **пропуск з патерном** — це вже категорія, а не шум.


In [0]:
skip = {"city", "snapshot_date"}
miss = (df.groupBy("city")
          .agg(*[F.avg(F.col(c).isNull().cast("int")).alias(c) for c in df.columns if c not in skip])
          .toPandas().set_index("city").T * 100)
miss = miss[CITY_ORDER].round(2)
mx = miss.max(axis=1)
miss["rule"] = np.select([mx == 0, mx < 1, mx <= 30], ["complete", "<1%: drop rows", "1–30%: impute + flag"],
                         default=">30%: why empty?")
miss = miss.sort_values(CITY_ORDER[0], ascending=False)
display(miss.reset_index().rename(columns={"index": "column"}))

for city in CITY_ORDER:
    KEY[f"price_missing_pct_{city}"] = float(miss.loc["price", city])


%md

- **9 колонок порожні на 100% в обох містах** (`host_since`, `bathrooms`, `host_response_*`, `instant_bookable`, …). Inside Airbnb більше не публікує ці поля у знімках 2026 року, тож їх видаляємо. Замість `bathrooms` беремо `bathrooms_text`, замість `host_since` — `hosts_time_as_host_*`.
- **`price` (9.4% / 10.7%) не імпутуємо**: це таргет, тож рядки без ціни виключаємо з train/test. 
- **Відгуки (11.9% / 24.8%)**: усі `review_scores_*`, `first/last_review` і `reviews_per_month` порожні в одних і тих самих рядках. Це оголошення без жодного відгуку, тобто пропуск з патерном, а не випадковий. Середнім не імпутуємо, а додаємо індикатор `no_reviews`. У Sicily таких оголошень удвічі більше.
- **`license` (14% / 20%)**: відсутність ліцензії сама по собі інформативна, тому робимо ознаку `has_license`, а не імпутуємо.
- **`minimum_nights`/`maximum_nights` (~3%) і `beds` (~1.8%)**: імпутуємо медіаною на train і додаємо флаг `is_missing`.
- **Менше 1% пропусків** (`bedrooms`, `bathrooms_text`, `has_availability`, `description`): ці рядки видаляємо або імпутуємо, на результат це майже не впливає.
- Похідні колонки (`log_price`, `long_stay`, `bathrooms_from_text`, `days_since_last_review`) просто успадковують пропуски своїх джерел.
- Rome і Sicily мають майже однакову структуру пропусків. Єдина помітна різниця — частка оголошень без відгуків.

### 5.1 Патерн №1 — коли відсутня ціна?
Гіпотеза: ціна порожня в **неактивних** оголошеннях (немає вільних дат, немає свіжих відгуків). Якщо так — це MNAR (missing not at random), і такі рядки треба виключати з навчання, а не імпутувати.


In [0]:
pm = (df.withColumn("price_missing", F.col("price").isNull())
        .groupBy("city", "price_missing").agg(
            F.count("*").alias("listings"),
            F.round(F.avg((F.col("availability_365") == 0).cast("int")), 3).alias("share_availability_365_eq_0"),
            F.percentile_approx("availability_365", 0.5).alias("median_availability_365"),
            F.round(F.avg((F.col("number_of_reviews_ltm") == 0).cast("int")), 3).alias("share_no_reviews_last_12m"),
            F.percentile_approx("days_since_last_review", 0.5).alias("median_days_since_last_review"),
            F.round(F.avg(F.col("has_availability").cast("int")), 3).alias("share_has_availability"),
            F.round(F.avg(F.col("review_scores_rating").isNull().cast("int")), 3).alias("share_no_rating"),
        ).orderBy("city", "price_missing"))
display(pm)


%md
**Висновок: ціна відсутня не випадково (MNAR).** Здебільшого це неактивні оголошення.

- Rome має 37 451 оголошення, з них без ціни 3 518 (9.4%). Sicily має 56 783 оголошення, без ціни 6 051 (10.7%).
- Серед оголошень без ціни немає жодного вільного дня на рік у 70% (Rome) і 49% (Sicily). Серед оголошень з ціною таких лише 3.7%. Медіана `availability_365` для оголошень без ціни — 0–1 день, для оголошень з ціною — 258–274 дні.
- Відгуків за 12 місяців немає у 59–70% оголошень без ціни, а з останнього відгуку минуло в медіані 348–387 днів, тобто близько року. Для оголошень з ціною ця медіана — 25–40 днів.
- `has_availability` майже однаковий в обох групах (0.97 проти 0.94), тому для визначення активності він не підходить. Краще дивитися на `availability_365` і давність останнього відгуку.

**Рішення:** рядки без ціни виключаємо з навчання й не імпутуємо. Інакше модель вчилася б на «мертвих» оголошеннях, які ніхто не бронює. У silver ці рядки лишаємо з флагом `has_price = false`, щоб дашборд правильно рахував загальну кількість оголошень.

**Наслідок для моделі:** вона описує ціни **активних** оголошень. Модель, навчена на Rome і Sicily, спиратиметься на цей фільтр, тому в Naples треба перевірити, чи частка неактивних оголошень там схожа.

### 5.2 Патерн №2 — відгуки, спальні, хост
* `review_scores_*` порожні ⇔ `number_of_reviews = 0`? Тоді це не пропуск, а категорія «нове оголошення».
* `bedrooms`/`beds` порожні залежно від `room_type` (студії, кімнати в hotel)?
* Поля хоста (`host_response_rate`, `host_response_time`, `host_acceptance_rate`) зникають **разом**?


In [0]:
display(df.groupBy("city",
                   F.when(F.col("number_of_reviews") == 0, "0 reviews").otherwise("≥1 review").alias("reviews"))
          .agg(F.count("*").alias("listings"),
               F.round(F.avg(F.col("review_scores_rating").isNull().cast("int")), 3).alias("rating_null_share"),
               F.round(F.avg(F.col("reviews_per_month").isNull().cast("int")), 3).alias("reviews_per_month_null_share"))
          .orderBy("city", "reviews"))

display(df.groupBy("city", "room_type")
          .agg(F.count("*").alias("listings"),
               *[F.round(F.avg(F.col(c).isNull().cast("int")), 3).alias(f"{c}_null")
                 for c in ["bedrooms", "beds", "bathrooms", "price"] if c in df.columns])
          .orderBy("city", F.desc("listings")))


%md
**Пропуски у відгуках — це категорія, а не втрачені дані.** Пропуск у `review_scores_rating` і `reviews_per_month` збігається з `number_of_reviews = 0` на **100%**: у Rome це 4 461 оголошення (11.9%), у Sicily 14 073 (24.8%). Серед оголошень, що мають хоча б один відгук, пропусків **нуль**. Тому середнім не імпутуємо, а додаємо ознаку `no_reviews`.

**Пропуски за `room_type`:**
- `bedrooms` майже повний (≤ 0.1%, лише в shared room у Sicily 1.5%, але там 67 оголошень). `beds` порожній в 1–3.4% рядків незалежно від типу житла. Імпутуємо медіаною за (`room_type`, `accommodates`) на train.
- `bathrooms` порожній на 100% для всіх типів, тому використовуємо `bathrooms_from_text`.
- Ціни найчастіше немає в `Hotel room` (18–23%) і `Shared room` (15–16%), а в `Entire home/apt` і `Private room` її немає в 9–11% оголошень. Але готелів і shared room дуже мало (0.1–0.5% оголошень), тож на модель це майже не впливає.

**Мікс типів житла:** у Rome `Entire home/apt` становить 74%, `Private room` — 25%. У Sicily `Entire home/apt` — 80%, `Private room` — 20%. На `Hotel room` і `Shared room` разом припадає менш ніж 1% оголошень. Це означає, що в baseline (`room_type` × `accommodates`) ці групи матимуть мало оголошень, тож для них потрібен fallback.

## 6. Ціна 
Five-number summary Тьюкі + середнє, skewness, kurtosis — на ціні та на `log(price)`.


In [0]:
priced = df.filter(F.col("price").isNotNull())
ps = (priced.groupBy("city").agg(
        F.count("price").alias("n"),
        F.min("price").alias("min"),
        F.percentile_approx("price", [0.25, 0.5, 0.75, 0.95, 0.99], 10000).alias("q"),
        F.max("price").alias("max"),
        F.round(F.mean("price"), 1).alias("mean"),
        F.round(F.skewness("price"), 2).alias("skew"),
        F.round(F.kurtosis("price"), 1).alias("kurtosis"),
        F.round(F.skewness("log_price"), 2).alias("skew_log"))
      .select("city", "n", "min", F.col("q")[0].alias("p25"), F.col("q")[1].alias("median"),
              F.col("q")[2].alias("p75"), F.col("q")[3].alias("p95"), F.col("q")[4].alias("p99"),
              "max", "mean", "skew", "kurtosis", "skew_log").orderBy("city"))
display(ps)
psd = ps.toPandas().set_index("city")
for city in CITY_ORDER:
    KEY[f"median_price_{city}"] = float(psd.loc[city, "median"])
    KEY[f"mean_price_{city}"] = float(psd.loc[city, "mean"])
    KEY[f"skew_price_{city}"] = float(psd.loc[city, "skew"])


In [0]:
# Гістограма log10(price): Spark рахує біни, на драйвер іде лише ~100 рядків
BW = 0.05
h = (priced.filter("price > 0")
       .withColumn("bin", F.floor(F.log10("price") / BW) * BW)
       .groupBy("city", "bin").count().toPandas())
h["share"] = h["count"] / h.groupby("city")["count"].transform("sum")
fig, ax = plt.subplots(figsize=(10, 4))
for city in CITY_ORDER:
    s = h[h.city == city].sort_values("bin")
    ax.step(10 ** s["bin"], s["share"], where="post", color=COLORS[city], lw=1.6)
    med = psd.loc[city, "median"]
    ax.axvline(med, color=COLORS[city], ls=":", lw=1)
    ax.text(med, s["share"].max() * 1.02, f"{city}: median {med:.0f}", color=COLORS[city], fontsize=9, ha="left")
ax.set_xscale("log"); ax.set_xlabel(f"price per night, {CURRENCY} (log scale)"); ax.set_ylabel("share of listings")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"Ціна скошена вправо (skew {psd.loc[CITY_ORDER[0],'skew']:.0f} / {psd.loc[CITY_ORDER[1],'skew']:.0f}), "
             f"а log(price) майже симетричний (skew {psd.loc[CITY_ORDER[0],'skew_log']:.1f} / {psd.loc[CITY_ORDER[1],'skew_log']:.1f}) — "
             "звітуємо медіану, моделюємо log", loc="left", fontsize=10.5)
plt.tight_layout(); plt.show()


%md
**Ціна.** Розподіл сильно скошений вправо: skew 30 у Rome і 113 у Sicily, а середнє на 35–50% вище за медіану (254 проти 188 € у Rome, 173 проти 115 € у Sicily). Тому звітуємо **медіану**, а модель навчаємо на **log(price)**: його skew лише 0.8 / 1.1, тобто розподіл майже симетричний.
- Rome у середньому дорожчий: медіана 188 € проти 115 € у Sicily. 50% оголошень у Rome коштують 134–273 €, у Sicily — 84–172 €.
- 99% оголошень коштують до ~1 000–1 200 €, а максимум сягає 37 474 € у Rome і 89 969 € у Sicily. Такі ціни найімовірніше є «заглушками», а не реальними цінами, тому їх розбираємо в розділі «Викиди».

In [0]:
# Box plot по room_type — квантилі рахує Spark, matplotlib лише малює (bxp)
bq = (priced.groupBy("city", "room_type")
        .agg(F.count("*").alias("n"), F.percentile_approx("price", [0.05, 0.25, 0.5, 0.75, 0.95], 10000).alias("q"))
        .toPandas())
rt_order = (bq.groupby("room_type")["n"].sum().sort_values(ascending=False).index.tolist())
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, city in zip(axes, CITY_ORDER):
    s = bq[bq.city == city].set_index("room_type").reindex([r for r in rt_order if r in bq[bq.city == city].room_type.values])
    stats = [dict(label=f"{rt}\n(n={int(r.n)})", whislo=r.q[0], q1=r.q[1], med=r.q[2], q3=r.q[3], whishi=r.q[4], fliers=[])
             for rt, r in s.iterrows()]
    ax.bxp(stats, showfliers=False, patch_artist=True,
           boxprops=dict(facecolor=COLORS[city], alpha=0.35), medianprops=dict(color=COLORS[city], lw=2))
    ax.set_yscale("log"); ax.set_title(city, loc="left"); ax.tick_params(axis="x", labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel(f"price, {CURRENCY} (log; whiskers = p5–p95)")
fig.suptitle("Rome дорожчий за Sicily в кожному типі житла", x=0.01, ha="left", fontsize=11)
plt.tight_layout(); plt.show()


### 6.1 Перевірка на Simpson's paradox
«Місто A дорожче за B» — чи це правда **всередині кожного room_type**, чи лише через інший мікс типів житла?


In [0]:
sim = (priced.groupBy("city", "room_type").agg(F.count("*").alias("n"), F.percentile_approx("price", 0.5).alias("median"))
         .toPandas())
sim["share"] = sim["n"] / sim.groupby("city")["n"].transform("sum")
piv = sim.pivot(index="room_type", columns="city", values=["median", "share"]).round(2)
piv.columns = [f"{v}_{c}" for v, c in piv.columns]
overall = psd["median"]
print("Загальна медіана:", overall.to_dict())
display(piv.reset_index())
a, b = CITY_ORDER
inside = (sim.pivot(index="room_type", columns="city", values="median").dropna())
same_sign = np.sign(inside[a] - inside[b]) == np.sign(overall[a] - overall[b])
print(f"Напрямок «{a} vs {b}» зберігається у {same_sign.sum()} з {len(same_sign)} room types"
      + ("" if same_sign.all() else " → агрегат вводить в оману, порівнюйте по room_type"))


%md
**Ціна за типом житла і перевірка на Simpson's paradox**
- `Entire home/apt` і `Hotel room` мають майже однакову медіану: 206 / 203 € у Rome і 123 / 119 € у Sicily. `Private room` дешевший приблизно в 1.4–1.5 раза, `Shared room` — приблизно в 4 рази. 
- **Rome дорожчий за Sicily в кожному `room_type`**, приблизно в 1.5–1.7 раза: entire home 206 проти 123 €, private room 139 проти 91 €, hotel room 203 проти 119 €, shared room 49 проти 33 €. Загальна різниця медіан (188 проти 115 €, тобто в 1.64 раза) має той самий напрямок і масштаб.
- Мікс типів житла в містах схожий: entire home становить 75% у Rome і 79% у Sicily. **Отже, Simpson's paradox тут немає**: Rome дорожчий не через інший мікс оголошень, а саме як ринок. Для моделі це означає, що місто (або рівень цін у ньому) — окремий сильний чинник.
- `Hotel room` (154 / 90 оголошень) і `Shared room` (83 / 57) мають мало даних, тому їхні медіани ненадійні, і для baseline потрібен fallback.
- Нулі в колонках `share` з'явилися через округлення до двох знаків: насправді частка `Hotel room` і `Shared room` разом менша за 1%.

## 7. Доступність (availability) і мінімальна тривалість


In [0]:
av = (df.withColumn("bin", F.floor(F.col("availability_365") / 15) * 15)
        .groupBy("city", "bin").count().toPandas())
av["share"] = av["count"] / av.groupby("city")["count"].transform("sum")
avs = (df.groupBy("city").agg(
          F.avg((F.col("availability_365") == 0).cast("int")).alias("zero"),
          F.avg((F.col("availability_365") >= 360).cast("int")).alias("full"),
          F.percentile_approx("availability_365", 0.5).alias("median"),
          F.avg(F.col("long_stay").cast("int")).alias("long_stay"))
         .toPandas().set_index("city"))
fig, ax = plt.subplots(figsize=(10, 3.8))
w = 6.5
for i, city in enumerate(CITY_ORDER):
    s = av[av.city == city].sort_values("bin")
    ax.bar(s["bin"] + 7.5 + (i - 0.5) * w, s["share"], width=w, color=COLORS[city], label=city)
ax.set_xlabel("availability_365 (days available in next year)"); ax.set_ylabel("share of listings")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
ax.set_title(f"Доступність бімодальна: {avs.loc[a,'zero']:.0%}/{avs.loc[b,'zero']:.0%} оголошень мають 0 вільних днів, "
             f"{avs.loc[a,'full']:.0%}/{avs.loc[b,'full']:.0%} — майже весь рік", loc="left", fontsize=11)
plt.tight_layout(); plt.show()
display(avs.round(3).reset_index())
for city in CITY_ORDER:
    KEY[f"availability_zero_share_{city}"] = round(float(avs.loc[city, "zero"]), 3)
    KEY[f"long_stay_share_{city}"] = round(float(avs.loc[city, "long_stay"]), 3)


%md
**Доступність (`availability_365`).** Розподіл бімодальний: є два піки, на краях року.
- **Пік біля 0**: 10% (Rome) і 9% (Sicily) оголошень мають рівно 0 вільних днів. Як показала таблиця пропусків, це здебільшого оголошення без ціни, тобто закриті або неактивні, а не «повністю заброньовані». Серед оголошень з ціною таких лише 3.7%.
- **Пік біля 365** різний у містах: у Sicily 21% оголошень відкриті майже весь рік (360+ днів), а в Rome лише 4%. Римський пік зсунутий на 300–345 днів, тобто частину дат там уже заброньовано. Сицилійський ринок має більше оголошень з порожнім календарем: сезонне житло, другі будинки, оголошення з низьким попитом поза сезоном (скрейп кінця вересня).
- Між піками розподіл рівномірний і низький (~2–3% на кожні 15 днів).

In [0]:
# ціна vs активність і long-stay — чи це різні ринки?
display(priced.groupBy("city",
            F.when(F.col("availability_365") == 0, "0").when(F.col("availability_365") < 90, "1–89")
             .when(F.col("availability_365") < 270, "90–269").otherwise("270–365").alias("availability_bucket"),
            "long_stay")
        .agg(F.count("*").alias("n"), F.percentile_approx("price", 0.5).alias("median_price"),
             F.percentile_approx("price_per_guest", 0.5).alias("median_price_per_guest"))
        .orderBy("city", "availability_bucket", "long_stay"))


%md
**Доступність і довгострокова оренда**
- **Long-stay — окремий ринок з іншою ціною за ніч.** Оголошення з `minimum_nights ≥ 28` дешевші за ніч: у Rome 84–110 € проти 178–200 € (приблизно вдвічі), у Sicily 82–85 € проти 120–124 € (приблизно на третину). Це ціна місячної оренди, перерахована на одну ніч. Таких оголошень 5% у Rome і 12% у Sicily (у Sicily здебільшого з календарем, відкритим майже весь рік). Тому `long_stay` — обов'язкова ознака моделі, інакше вона завищуватиме ціну для таких оголошень.
- **Пропуск `minimum_nights` має патерн.** Усі 976 (Rome) і 1 855 (Sicily) оголошень, де `long_stay = null`, мають 0 вільних днів. Тобто `minimum_nights` порожній саме тоді, коли календар закритий. Як і у випадку з ціною, це неактивні оголошення, тож імпутувати медіаною тут неправильно, краще додати флаг.
- **Доступність майже не впливає на ціну активних оголошень.** Серед оголошень без long-stay ціна в Rome трохи зростає з доступністю (178 → 192 → 200 €), а в Sicily не змінюється (~120 €). Оголошення з 0 вільних днів у Rome дешевші (135–160 €).
- `price_per_guest` показано тільки для аналізу. Він обчислюється з ціни, тому як ознака моделі це leakage.

## 8. Інші ключові ознаки: accommodates, neighbourhoods, review scores


In [0]:
# Медіана ціни по (room_type, accommodates) — це і є baseline з завдання. Скільки клітинок замалі?
cell = (priced.groupBy("city", "room_type", "accommodates")
          .agg(F.count("*").alias("n"), F.percentile_approx("price", 0.5).alias("median")).toPandas())
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
main_rt = rt_order[:2]
for ax, city in zip(axes, CITY_ORDER):
    for rt in rt_order:
        s = cell[(cell.city == city) & (cell.room_type == rt) & (cell.n >= 20)].sort_values("accommodates")
        hi = rt in main_rt
        ax.plot(s.accommodates, s["median"], marker="o", ms=3, lw=2 if hi else 1,
                color=COLORS[city] if rt == main_rt[0] else ("#333" if hi else GREY))
        if len(s): ax.text(s.accommodates.iloc[-1] + 0.2, s["median"].iloc[-1], rt, fontsize=8,
                           color=COLORS[city] if rt == main_rt[0] else ("#333" if hi else GREY))
    ax.set_title(city, loc="left"); ax.set_xlabel("accommodates"); ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel(f"median price, {CURRENCY}")
fig.suptitle("Ціна росте майже лінійно з кількістю гостей — baseline (room_type × accommodates) вже сильний", x=0.01, ha="left", fontsize=11)
plt.tight_layout(); plt.show()

small = cell.assign(small=cell.n < 5).groupby("city").agg(cells=("n", "size"), small_cells=("small", "sum"),
                                                           listings_in_small=("n", lambda s: s[s < 5].sum()))
display(small.reset_index())
print("→ для baseline потрібен fallback (наприклад, медіана room_type), коли клітинка порожня або < 5 оголошень.")


%md
**Ціна × кількість гостей × тип житла**
- **Rome, entire home:** ціна зростає майже лінійно, приблизно на 55 € за кожного додаткового гостя: від ~100 € (1 гість) до ~940 € (16 гостей).
- **Sicily, entire home:** до 5 гостей ціна майже не змінюється (~90–130 €), а з 6 гостей різко росте (до ~800 € за 16 гостей). Великі оголошення в Sicily — це вілли, тобто окремий, дорожчий сегмент. Крива шумна (наприклад, провал на 15 гостях), бо великих оголошень мало.
- **Private room:** у Rome ціна росте до 5 гостей (~210 €), у Sicily тримається на рівні ~100 € незалежно від кількості гостей. Отже, вплив розміру залежить від міста і типу житла.
- **Малі оголошення (1–4 гості)** у Sicily коштують майже однаково для entire home і private room (~90–110 €), а в Rome entire home помітно дорожчий. Тобто в Sicily `room_type` важить менше, ніж у Rome.
- `Hotel room` і `Shared room` є лише для 1–3 гостей і мають мало даних. 

In [0]:
nb = (priced.groupBy("city", "neighbourhood_cleansed")
        .agg(F.count("*").alias("n"), F.percentile_approx("price", 0.5).alias("median")).toPandas())
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, city in zip(axes, CITY_ORDER):
    s = nb[(nb.city == city) & (nb.n >= 30)].sort_values("median")
    s = pd.concat([s.head(8), s.tail(8)]).drop_duplicates()
    cm_ = psd.loc[city, "median"]
    ax.barh(s.neighbourhood_cleansed, s["median"], color=[COLORS[city] if v > cm_ else GREY for v in s["median"]])
    ax.axvline(cm_, color="#333", lw=1, ls=":"); ax.text(cm_, -0.9, f" city median {cm_:.0f}", fontsize=8)
    ax.set_title(f"{city}: {nb[nb.city==city].shape[0]} neighbourhoods" + (" (муніципалітети)" if city == "sicily" else ""), loc="left", fontsize=10)
    ax.tick_params(axis="y", labelsize=8); ax.spines[["top", "right"]].set_visible(False)
spread = {c: nb[(nb.city == c) & (nb.n >= 30)]["median"] for c in CITY_ORDER}
if all(len(v) for v in spread.values()): fig.suptitle("Найдорожчий район у " + " / ".join(f"{c}: {spread[c].max()/spread[c].min():.1f}×" for c in CITY_ORDER)
             + " дорожчий за найдешевший (райони з n ≥ 30)", x=0.01, ha="left", fontsize=11)
plt.tight_layout(); plt.show()


%md
**Локація**
- **Rome:** лише `I Centro Storico` має медіану (~235 €), вищу за медіану міста (188 €). Медіани всіх інших 14 муніципіїв нижчі (113–175 €). Медіана міста вища, ніж у 14 з 15 районів, тому що в центрі зосереджена велика частка оголошень, і вона тягне загальну медіану вгору. 
- **Sicily:** найдорожчі комуни — туристичні курорти та острови (Taormina, Santa Marina Salina на Еолійських островах, узбережжя біля Палермо й Трапані), ~160–200 €. Найдешевші — внутрішні міста (Caltanissetta, Enna, Gangi, Canicattì), ~58–78 €. Різниця становить до 3.4 раза проти 2.1 раза в Rome.
- Отже, локація — сильний чинник ціни, але працює по-різному: у Rome ціну визначає близькість до центру, у Sicily — узбережжя чи острови проти внутрішніх районів.

## 9. Викиди: виявлення і класифікація 

IQR на сирій ціні (на скошених даних позначає хвіст), IQR і modified z-score на `log(price)`, доменні правила.
**Outlier ≠ error**: кожен позначений рядок отримує клас, і для класу — рішення.


In [0]:
rows = []
bounds = {}
for city in CITY_ORDER:
    s = priced.filter((F.col("city") == city) & (F.col("price") > 0))
    n = s.count()
    q1, q3 = s.approxQuantile("price", [0.25, 0.75], 0.001)
    lo_raw, hi_raw = q1 - IQR_K * (q3 - q1), q3 + IQR_K * (q3 - q1)
    l1, l3 = s.approxQuantile("log_price", [0.25, 0.75], 0.001)
    lo_log, hi_log = l1 - IQR_K * (l3 - l1), l3 + IQR_K * (l3 - l1)
    med = s.approxQuantile("log_price", [0.5], 0.001)[0]
    mad = s.select(F.abs(F.col("log_price") - med).alias("d")).approxQuantile("d", [0.5], 0.001)[0]
    bounds[city] = dict(lo_log=lo_log, hi_log=hi_log, med=med, mad=mad)
    rows.append(dict(city=city, n=n,
        iqr_raw_flagged=s.filter((F.col("price") < lo_raw) | (F.col("price") > hi_raw)).count(),
        iqr_raw_fence=f"[{lo_raw:.0f}, {hi_raw:.0f}]",
        iqr_log_flagged=s.filter((F.col("log_price") < lo_log) | (F.col("log_price") > hi_log)).count(),
        iqr_log_fence=f"[{np.exp(lo_log):.0f}, {np.exp(hi_log):.0f}]",
        mz_log_flagged=s.filter(F.abs(0.6745 * (F.col("log_price") - med) / F.lit(mad)) > MZ_THRESHOLD).count()))
out = pd.DataFrame(rows)
out["iqr_raw_%"] = (100 * out.iqr_raw_flagged / out.n).round(1)
out["iqr_log_%"] = (100 * out.iqr_log_flagged / out.n).round(1)
display(out)
print("IQR на сирій скошеній ціні позначає значно більше рядків — це хвіст, а не помилки. Далі працюємо на log(price).")


%md
**Викиди: виявлення і класифікація**

*Порівняння методів*
- **IQR на сирій ціні не підходить:** нижня межа від'ємна (−73 / −48 €), тобто цін-викидів знизу він не знайде в принципі. Зате позначає 6.8% (Rome) і 9.4% (Sicily) оголошень зверху, а це просто довгий хвіст скошеного розподілу, як і попереджала лекція.
- **IQR на log(price)** дає осмислені межі: 47–789 € (Rome) і 29–505 € (Sicily). Позначає 3.2% / 4.2% оголошень, причому з обох боків.
- **Modified z-score на log(price)** (поріг 3.5) найконсервативніший: 415 (1.2%) і 999 (2.0%) оголошень.
- Межі рахуємо **окремо для кожного міста**: сицилійська верхня межа (505 €) для Rome була б нормальною ціною.


In [0]:
def bexpr(key):
    return F.create_map(*[x for c in CITY_ORDER for x in (F.lit(c), F.lit(bounds[c][key]))])[F.col("city")]

mz = 0.6745 * (F.col("log_price") - bexpr("med")) / bexpr("mad")
inactive = (F.col("availability_365") == 0) & (F.coalesce(F.col("number_of_reviews_ltm"), F.lit(0)) == 0)

classified = (df.withColumn("mz_log_price", mz)
    .withColumn("outlier_class", F
        .when(F.col("price").isNull(), "no_price")
        .when((F.col("price") <= 0) | (F.col("accommodates") <= 0) | (F.col("accommodates") > MAX_GUESTS)
              | (F.col("minimum_nights") > F.col("maximum_nights")) | F.col("latitude").isNull(), "error")
        .when((F.col("price") >= MAX_PRICE) | (F.col("price") < MIN_PRICE)
              | ((F.abs(F.col("mz_log_price")) > MZ_THRESHOLD) & inactive), "suspicious")
        .when((F.col("log_price") < bexpr("lo_log")) | (F.col("log_price") > bexpr("hi_log")), "tail")
        .otherwise("normal")))

oc = (classified.groupBy("city", "outlier_class").agg(
        F.count("*").alias("n"),
        F.percentile_approx("price", 0.5).alias("median_price"),
        F.round(F.avg(inactive.cast("int")), 3).alias("share_inactive"),
        F.round(F.avg(F.col("long_stay").cast("int")), 3).alias("share_long_stay"))
      .orderBy("city", "outlier_class"))
display(oc)
ocd = oc.toPandas()
for city in CITY_ORDER:
    for k in ["error", "suspicious", "tail"]:
        v = ocd[(ocd.city == city) & (ocd.outlier_class == k)]["n"]
        KEY[f"outliers_{k}_{city}"] = int(v.iloc[0]) if len(v) else 0


*Класи (доменні правила + log IQR)*
| Клас | Rome | Sicily | Що це | Рішення |
|---|---|---|---|---|
| `error` (ціна ≤ 0, accommodates ∉ 1–16, min > max nights, немає координат) | 0 | 0 | у даних немає явних помилок | — |
| `suspicious` (≥ 5 000 € або < 10 € за ніч, або екстремальна ціна в неактивного оголошення) | 49, медіана 8 426 € | 32, медіана 5 655 € | ціни-«заглушки», якими хост блокує календар | **виключаємо** з train/test, у silver лишаємо з флагом |
| `tail` (поза log IQR, але правдоподібно) | 1 049, медіана 926 € | 2 117, медіана 672 € | вілли, великі апартаменти, частково long-stay | **лишаємо**: log(price) зменшує їхній вплив |
| `no_price` | 3 518 | 6 051 | неактивні оголошення (розд. 5.1) | виключаємо: немає таргету |

In [0]:
cols_show = ["city", "id", "name", "room_type", "accommodates", "bedrooms", "price", "minimum_nights",
             "availability_365", "number_of_reviews_ltm", "outlier_class"]
display(classified.filter("price is not null").orderBy(F.desc("price")).select(cols_show).limit(25))
display(classified.filter("price is not null").orderBy("price").select(cols_show).limit(15))


%md
**Ручний перегляд крайніх цін**

*Найдорожчі (`suspicious`, ≥ 5 000 € за ніч) — це не реальні ціни за ніч:*
- **Ціни-заглушки в неактивних оголошеннях.** Чотири майже ідентичні «Marconi B&B» (одна назва, ~13 000 €) мають `availability_365 = 0` і порожній `minimum_nights`. Це закриті календарі, плюс приклад «близнюків» одного хоста. Округлені 9 998 € і 10 000 € теж схожі на заглушку.

→ **Виключаємо всі `suspicious` з train/test** (81 оголошення). Поріг 5 000 € робить те, що потрібно.

*Найдешевші (9.85–15 €) — майже всі з Sicily і майже всі з `minimum_nights = 31`:*
- Це **довгострокова оренда, ціна якої перерахована на ніч**: 13 € × 31 ≈ 410 € на місяць за кімнату. Ціна реальна, але це інший ринок, а не туристичні ночі.



## 10. Зв'язки з ціною



In [0]:
clean = classified.filter(F.col("outlier_class").isin("normal", "tail"))
num_cols = [c for c, t in clean.dtypes if t in ("double", "bigint", "int", "boolean")
            and c not in ("id", "host_id", "price", "log_price", "mz_log_price", "price_per_guest")]
n_city = {r["city"]: r["count"] for r in clean.groupBy("city").count().collect()}
frac = {c: min(1.0, SAMPLE_N / n) for c, n in n_city.items()}
pdf = (clean.sampleBy("city", frac, seed=SEED)
            .select("city", "log_price", *[F.col(c).cast("double").alias(c) for c in num_cols]).toPandas())

rho = pd.DataFrame({city: g.drop(columns="city").corr(method="spearman")["log_price"].drop("log_price")
                    for city, g in pdf.groupby("city")})[CITY_ORDER]
rho = rho.reindex(rho.abs().max(axis=1).sort_values(ascending=False).index)
display(rho.round(3).reset_index().rename(columns={"index": "feature"}))

top = rho.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(top))
for i, city in enumerate(CITY_ORDER):
    ax.barh(y + (i - 0.5) * 0.4, top[city], height=0.4, color=COLORS[city], label=city)
ax.set_yticks(y, top.index, fontsize=8); ax.axvline(0, color="#333", lw=0.8)
ax.set_xlabel("Spearman ρ with log(price)"); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
same = (np.sign(rho[CITY_ORDER[0]]) == np.sign(rho[CITY_ORDER[1]])).head(10).mean()
ax.set_title(f"Найсильніше з log(price) пов'язана {top.index[-1]} (ρ≈{top.iloc[-1].mean():.2f}); "
             f"у топ-10 ознак напрямок збігається в обох містах для {same:.0%}", loc="left", fontsize=11)
plt.tight_layout(); plt.show()
suspicious_corr = rho[rho.abs().max(axis=1) > 0.9]
print("⚠ |ρ| > 0.9 з ціною (кандидати на leakage):", suspicious_corr.index.tolist())


%md
**Зв'язок ознак з ціною (Spearman ρ з log_price, на оголошеннях `normal` + `tail`)**

- **Розмір житла — найсильніший чинник в обох містах:** `accommodates` (0.50 / 0.42), `bedrooms` (0.43 / 0.42), `bathrooms_from_text` (0.43 / 0.41), `beds` (0.40 / 0.34). Ці ознаки сильно корелюють між собою, тобто описують одне й те саме — «наскільки велике житло». Для моделі достатньо 2–3 з них, а для лінійної моделі це мультиколінеарність.
- **Знижують ціну:** спільна ванна (`bathrooms_shared`, −0.23 / −0.13) і довгострокова оренда (`long_stay`, −0.20 / −0.22). Обидві ознаки працюють однаково в обох містах.
- **Професійні хости дорожчі:** чим більше в хоста цілих квартир (`calculated_host_listings_count_entire_homes`, 0.34 / 0.24), тим вища ціна. З кімнатами навпаки (−0.20 / −0.15).
- **Локація працює по-різному:**
  - `review_scores_location`: 0.39 у Rome, але лише 0.10 у Sicily. У Rome оцінка локації фактично означає «близько до центру».
  - `latitude`: 0.19 у Rome, 0.00 у Sicily. Spearman ловить лише монотонний зв'язок. У Sicily дорогі курорти розкидані по всьому узбережжю й островах, тому сира широта нічого не пояснює. Локацію треба кодувати інакше: відстань до центру (Rome) або до узбережжя / туристичних центрів (Sicily). 
- **Попит і активність майже не пов'язані з ціною** (|ρ| < 0.1): `number_of_reviews*`, `availability_*`, `estimated_occupancy_l365d`, `reviews_per_month`. Те саме з характеристиками хоста (`host_is_superhost`, стаж, верифікація). 
- Для головних ознак напрямок зв'язку однаковий в обох містах, а сила в Sicily здебільшого менша. Ринок там неоднорідніший, тож у Sicily варто очікувати більшої похибки моделі.

## 11. Leakage review
Питання для кожної колонки: **(1)** чи її обчислено з ціни? **(2)** чи вона буде відома в момент, коли модель пропонує ціну *новому* оголошенню? **(3)** чи вона дозволяє моделі «впізнати» те саме оголошення/хоста в test?


In [0]:
# (1) estimated_revenue_l365d ≈ price × estimated_occupancy_l365d ?
if {"estimated_revenue_l365d", "estimated_occupancy_l365d"} <= set(df.columns):
    lk = (priced.filter("estimated_occupancy_l365d > 0")
          .withColumn("implied_price", F.col("estimated_revenue_l365d") / F.col("estimated_occupancy_l365d"))
          .withColumn("rel_err", F.abs(F.col("implied_price") - F.col("price")) / F.col("price")))
    res = lk.groupBy("city").agg(F.count("*").alias("n"),
                                 F.round(F.avg((F.col("rel_err") < 0.01).cast("int")), 4).alias("share_revenue_div_occupancy_eq_price"),
                                 F.round(F.corr("estimated_revenue_l365d", "price"), 3).alias("corr_revenue_price"))
    display(res)
    for r in res.collect():
        KEY[f"revenue_eq_price_x_occupancy_{r['city']}"] = r["share_revenue_div_occupancy_eq_price"]
else:
    print("estimated_revenue_l365d / estimated_occupancy_l365d відсутні в цьому знімку")


%md
** `estimated_revenue_l365d` обчислено з ціни.** У **100%** оголошень з ненульовою зайнятістю (Rome — 26 280, Sicily — 28 693) виконується `estimated_revenue_l365d / estimated_occupancy_l365d = price` з точністю до 1%. Тобто Inside Airbnb рахує revenue як `price × occupancy`. Якщо дати цю колонку моделі разом із `estimated_occupancy_l365d`, модель просто відновить ціну діленням, і MAE буде неправдоподібно низьким. **Колонку виключаємо з ознак.**

Pearson між revenue і ціною лише 0.66 (Rome) і 0.45 (Sicily), тож за кореляцією цей витік не помітний. Його видно, лише коли перевірити, як колонку обчислено.

## 12. Очищення і трансформації, які мусить робити пайплайн (Step 5)

### A. Bronze → silver: завантаження і типізація (для кожного міста, параметр `city`)

| # | Трансформація | Причина (що знайдено в EDA) |
|---|---|---|
| 1 | Читати CSV з `multiLine=True, escape='"'` | `description` і `host_about` містять переноси рядків і лапки; без цих опцій рядки ламаються (розд. 2) |
| 2 | Типи задавати **явно** за списком колонок (`try_cast`), а не довіряти `inferSchema`; `id`, `host_id` → `BIGINT` | `id` сягає ~1.8·10¹⁸: якщо `inferSchema` в іншому місті визначить `double`, ключі зламаються. У Rome і Sicily типи збіглися, але для Naples це не гарантовано (розд. 2) |
| 3 | Перевірка grain: `id` не null, числовий, унікальний; інакше — зупинити пайплайн або відкласти рядки в карантин | один рядок = одне оголошення; дублікати зламали б join'и і метрики (розд. 3) |
| 4 | Брати **явний список потрібних колонок**; відсутню колонку додавати як null, а не падати | 13 колонок порожні на 100% (`host_since`, `bathrooms`, `host_response_*`, `instant_bookable`, `neighbourhood`, …), а в старих знімках є інші колонки, ніж у знімках 2026 (розд. 3, 5) |
| 5 | `price`: прибрати все, крім цифр і крапки → `DOUBLE` | `price` — рядок `"$1,000.00"`; валюта насправді EUR, попри `$` (розд. 4) |
| 6 | `t`/`f` → `BOOLEAN` | `host_is_superhost`, `has_availability` тощо — рядки (розд. 4) |
| 7 | `bathrooms` ← число з `bathrooms_text` (`"Half-bath"` → 0.5); `bathrooms_shared` = текст містить «shared» | `bathrooms` порожній на 100%; з тексту кількість відома для 99.9% оголошень (розд. 4) |
| 8 | `host_tenure_months` = `hosts_time_as_host_years·12 + hosts_time_as_host_months` | `host_since` порожній на 100%, у знімку 2026 стаж хоста зберігається в нових колонках (розд. 3) |
| 9 | `review_scores_*` поза діапазоном [1, 5] → null | у Sicily є `0.0` при шкалі 1–5 (розд. 3) |
| 10 | `has_price = price IS NOT NULL`; рядки без ціни лишаються в silver | ціни немає в 9.4% / 10.7%; це неактивні оголошення (у 70% / 49% з них 0 вільних днів), тобто MNAR (розд. 5.1). Для дашборду (кількість оголошень, хостів) вони потрібні |
| 11 | `no_reviews = number_of_reviews = 0`; `review_scores_*`, `reviews_per_month`, `first/last_review` **не імпутувати** | пропуски в них збігаються з `number_of_reviews = 0` на 100%: 11.9% / 24.8% оголошень (розд. 5.2) |
| 12 | `long_stay = minimum_nights ≥ 28`; якщо `minimum_nights` null → `long_stay = false` + флаг `min_nights_missing` | long-stay — інший ринок: у Rome ціна за ніч ~удвічі нижча, у Sicily ~на третину; таких оголошень 5% / 12%. `minimum_nights` порожній лише в оголошеннях з 0 вільних днів (розд. 7) |
| 13 | `has_license`: null / `Exempt` / код | ліцензії немає в 14% / 20% оголошень, і сама відсутність ліцензії інформативна (розд. 5) |
| 14 | `amenities_count` = розмір JSON-масиву `amenities` | розд. 4 |
| 15 | `outlier_class` — **один клас на рядок, за пріоритетом** (спрацьовує перше правило): `no_price` → **`error`** (ціна ≤ 0, `accommodates` ∉ [1, 16], `minimum_nights > maximum_nights`, немає координат) → **`suspicious`** (`price ≥ 5 000` або `price < 10` і не long-stay) → **`tail`** (поза log-IQR міста, але не suspicious) → `normal` | заглушки до 90 тис. € за ніч (81 оголошення); дешеві long-stay — реальна місячна оренда, а не помилка; `error` у Rome/Sicily не знайшлося, але правило лишаємо як захист для нових міст (розд. 9) |

### B. Silver → gold: підготовка ознак і таргету для моделі

| # | Трансформація | Причина |
|---|---|---|
| 16 | У train/test беремо лише `has_price` і `outlier_class IN ('normal', 'tail')` | без таргету вчитися неможливо; заглушки зіпсували б MAE (розд. 5.1, 9) |
| 17 | Таргет `log(price)`; прогноз → `exp()` для MAE в EUR | skew ціни 30 / 113, skew log(price) — 0.8 / 1.1 (розд. 6) |
| 18 | Split 80/20 **окремо для кожного міста, згрупований за `host_id`** | 31–33% оголошень мають «близнюка» того самого хоста, 1.6–3.7% — з ідентичною ціною; випадковий split дав би оптимістичний MAE (розд. 11) |
| 19 | **Виключити leakage:** `estimated_revenue_l365d`, `price_quote_*` (`price_per_night`, `total_price`, `raw`, `checkin/checkout_date`), будь-які похідні від ціни (`price_per_guest`) | revenue ÷ occupancy = price у 100% оголошень; `price_quote_*` — та сама ціна з котирування (розд. 11) |
| 20 | **Не використовувати** ознаки попиту: `number_of_reviews*`, `reviews_per_month`, `review_scores_*`, `availability_*`, `estimated_occupancy_l365d`, `days_since_last_review` | вони є наслідком ціни, для нового оголошення невідомі, а з log(price) майже не корелюють (\|ρ\| < 0.1), тож втрата мала (розд. 10) |
| 21 | **Не використовувати** ідентифікатори й константи знімка: `id`, `host_id`, URL-колонки, `name`, `description`, `scrape_id`, `last_scraped`, `snapshot_date` | модель запам'ятовувала б конкретні оголошення, а константи знімка — лише проксі міста (розд. 11) |
| 22 | Локація: `latitude`, `longitude` + відстань до центру міста (координати центру — параметр міста). `neighbourhood_cleansed` **не one-hot**; target encoding — лише на train | 15 муніципіїв у Rome проти 376 комун у Sicily, у Naples будуть нові назви; у Sicily широта не корелює з ціною (ρ = 0), бо дорогі курорти розкидані по узбережжю (розд. 8, 10) |
| 23 | `property_type`: типи з < 50 оголошеннями → `Other` | 64 / 104 типи з довгим хвостом (Yurt, Windmill, Boat) (розд. 3) |
| 24 | `beds` (1.8% пропусків), `bedrooms` і `bathrooms` (< 0.1%) → медіана за (`room_type`, `accommodates`) **на train** + флаг `is_missing` | імпутація на всіх даних — leakage з test (розд. 5) |
| 25 | Ознаки розміру: `accommodates`, `bedrooms`, `bathrooms`, `beds` — найсильніші, але між собою сильно корелюють | ρ 0.34–0.50 з log(price) в обох містах; для лінійної моделі — мультиколінеарність (розд. 10) |
| 26 | Усі пороги (`MIN_PRICE`, `MAX_PRICE`, межі IQR) — параметри, межі IQR рахуються **окремо для кожного міста** | рівні цін різні: log-IQR межа 789 € у Rome проти 505 € у Sicily (розд. 9) |

## 13. Ключові цифри для підсумку і паспорт запуску (Step 8)


In [0]:
try:
    nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
except Exception:
    nb_path = "n/a"
header = pd.DataFrame([
    ("Source", "; ".join(c["url"] for c in CITIES)),
    ("Tables", "; ".join(f"{c['table']} (Delta v{c['delta_version']})" for c in CITIES)),
    ("Snapshots", "; ".join(f"{c['city']}={c['snapshot']}" for c in CITIES)),
    ("EDA table", T("eda_silver_listings_typed")),
    ("Rows", "; ".join(f"{c}={KEY[f'rows_{c}']}" for c in CITY_ORDER)),
    ("Schema", f"workspace.{schema}"),
    ("Run", f"{datetime.datetime.now():%Y-%m-%d %H:%M}, {nb_path}, seed={SEED}"),
], columns=["field", "value"])
display(header)
display(pd.DataFrame(sorted(KEY.items()), columns=["key", "value"]).astype(str))
